<a href="https://colab.research.google.com/github/sondosafify/Abn_Albld_Masry-AI-Chatbot/blob/main/chat_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install langchain-groq langchain-core
!pip install langchain-community langchain-groq langchain
!pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-

In [ ]:
# امسحي السطر القديم اللي كان فيه المفتاح gsk_xxx
# واكتبي السطر ده مكانه:
import os
import streamlit as st

if "GROQ_API_KEY" in st.secrets:
    os.environ["GROQ_API_KEY"] = st.secrets["GROQ_API_KEY"]
else:
    st.error("المفتاح مش موجود في إعدادات الموقع (Secrets)!")

In [4]:
# !pip install langchain-community langchain-groq langchain

import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from google.colab import userdata
import os

# سحب المفتاح من خزنة كولاب
api_key = userdata.get('GROQ_API_KEY')

# استخدامه في مشروعك
os.environ["GROQ_API_KEY"] = api_key
class MasryChatbot:
    def __init__(self):
        # تم تحديث الموديل لـ llama-3.3-70b-versatile عشان يتجنب خطأ الـ decommissioned
        self.llm = ChatGroq(
            model_name="llama-3.3-70b-versatile",
            temperature=0.8
        )

        self.store = {}

        # 2. هندسة الرد المصري
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", (
                "أنت مساعد ذكي ولطيف اسمك 'صاحبي'. "
                "تتحدث اللهجة العامية المصرية بطلاقة وخفة دم. "
                "ردودك لازم تكون بالمصري، ودودة، ومحترمة. "
                "لو حد سألك عن حاجة متعرفهاش، قوله بصراحة بس بلطافة."
            )),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{input}"),
        ])

        # 3. بناء الهيكل
        self.chain = self.prompt | self.llm

    def get_session_history(self, session_id: str):
        if session_id not in self.store:
            self.store[session_id] = ChatMessageHistory()
        return self.store[session_id]

    def ask(self, user_input: str, session_id: str = "user_1"):
        try:
            if not user_input.strip():
                return "مبعتليش حاجة ليه يا غالي؟"

            # استخدام الكلاس ده بيضمن إن الذاكرة تفضل شغالة طول ما البرنامج مفتوح
            with_message_history = RunnableWithMessageHistory(
                self.chain,
                self.get_session_history,
                input_messages_key="input",
                history_messages_key="history",
            )

            response = with_message_history.invoke(
                {"input": user_input},
                config={"configurable": {"session_id": session_id}}
            )
            return response.content
        except Exception as e:
            # لو الـ API Key فيه مشكلة أو الموديل مش متاح هيظهرلك هنا
            if "model_decommissioned" in str(e):
                return "الموديل ده قديم، غيرتلك الموديل لواحد أحدث، جرب تاني!"
            print(f"Error details: {e}")
            return "حصلت لخبطة بسيطة، ممكن تجرب تبعت تاني؟"

# --- التشغيل ---
bot = MasryChatbot()
print("--- 'صاحبي' رجع وبقوة! (اكتب 'خروج' للإنهاء) ---")

while True:
    user_msg = input("أنت: ")
    if user_msg.lower() in ['خروج', 'exit', 'quit']:
        print("صاحبي: نورتني يا غالي، في رعاية الله!")
        break

    answer = bot.ask(user_msg)
    print(f"صاحبي: {answer}")

--- 'صاحبي' رجع وبقوة! (اكتب 'خروج' للإنهاء) ---
أنت: خروج
صاحبي: نورتني يا غالي، في رعاية الله!
